In [2]:
# %% [code]
!pip install -q stardist csbdeep zarr geff networkx scipy scikit-image numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 96.9 MB/s eta 0:00:00:00:0100:01


In [3]:
# imports
import os
import numpy as np
import zarr
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.registration import phase_cross_correlation
from stardist.models import StarDist3D
from csbdeep.utils import normalize

# 1. Setup GPU Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load StarDist3D (Downloads weights on first run)
print("Loading StarDist3D...")
stardist_model = StarDist3D.from_pretrained('3D_demo')
print("StarDist3D loaded successfully!")

Using device: cuda
Loading StarDist3D...
Found model '3D_demo' for 'StarDist3D'.
5749101/5749101 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


I0000 00:00:1789445009.362198      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789445009.365090      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.707933, nms_thresh=0.3.
StarDist3D loaded successfully!


In [ ]:
# features
def preprocess_3d_volume(vol, alpha=1.2, sigma=(0.8, 1.5, 1.5)):
    """Applies 3D edge detection and unsharp masking to sharpen cell boundaries."""
    vol_float = vol.astype(np.float32)
    blurred = ndi.gaussian_filter(vol_float, sigma=sigma)
    edges = vol_float - blurred
    sharpened = vol_float + alpha * edges
    return np.clip(sharpened, 0, None)

def get_motion_vector(centroid, label_id, labels_T, frame_T, frame_T1, r=10):
    """Calculates 3D optical flow/shift for a specific cell using masked cross-correlation."""
    z, y, x = int(centroid[0]), int(centroid[1]), int(centroid[2])
    Z_max, Y_max, X_max = frame_T.shape

    z_min, z_max = max(0, z-r), min(Z_max, z+r)
    y_min, y_max = max(0, y-r), min(Y_max, y+r)
    x_min, x_max = max(0, x-r), min(X_max, x+r)

    crop_T = frame_T[z_min:z_max, y_min:y_max, x_min:x_max].copy()
    mask_crop = labels_T[z_min:z_max, y_min:y_max, x_min:x_max]
    
    # Zero out background / neighbor cells
    crop_T[mask_crop != label_id] = 0

    # Avoid warnings if the cell mask is entirely empty (edge of frame)
    if np.all(crop_T == 0):
        return np.array([0.0, 0.0, 0.0])

    crop_T1 = frame_T1[z_min:z_max, y_min:y_max, x_min:x_max]

    if crop_T.shape == crop_T1.shape and 0 not in crop_T.shape:
        shift, _, _ = phase_cross_correlation(crop_T, crop_T1, upsample_factor=10)
        return shift
    else:
        return np.array([0.0, 0.0, 0.0])

In [ ]:
# deep network
# =======================================
# 1. Appearance Embedding Network
# =======================================
class CellReID3D(nn.Module):
    def __init__(self, emb_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=3, padding=1), nn.BatchNorm3d(16), nn.ReLU(inplace=True), nn.MaxPool3d(2),
            nn.Conv3d(16, 32, kernel_size=3, padding=1), nn.BatchNorm3d(32), nn.ReLU(inplace=True), nn.AdaptiveAvgPool3d((2, 2, 2))
        )
        self.fc = nn.Linear(32 * 2 * 2 * 2, emb_dim)

    def forward(self, x):
        feat = self.conv(x)
        feat = feat.view(feat.size(0), -1)
        return F.normalize(self.fc(feat), p=2, dim=1) # L2 normalize for Cosine Distance

reid_model = CellReID3D(emb_dim=64).to(device)
reid_model.eval()

def extract_cell_embeddings(volume, centroids, crop_size=16):
    if len(centroids) == 0: return np.empty((0, 64))
    r = crop_size // 2
    vol_padded = np.pad(volume, r, mode='reflect')
    crops = []
    
    for c in centroids:
        cz, cy, cx = int(round(c[0])) + r, int(round(c[1])) + r, int(round(c[2])) + r
        crops.append(vol_padded[cz-r:cz+r, cy-r:cy+r, cx-r:cx+r])
        
    batch = torch.tensor(np.stack(crops), dtype=torch.float32).unsqueeze(1).to(device)
    batch = (batch - batch.mean(dim=(2,3,4), keepdim=True)) / (batch.std(dim=(2,3,4), keepdim=True) + 1e-6)
    
    with torch.no_grad():
        embeddings = reid_model(batch)
    return embeddings.cpu().numpy()

# =======================================
# 2. DeepSORT Tracker
# =======================================
class DeepTrack3D:
    def __init__(self, track_id, centroid, label_id, embedding):
        self.track_id = track_id
        self.current_centroid = centroid
        self.current_label = label_id
        self.motion_vector = np.array([0.0, 0.0, 0.0])
        self.features = [embedding] # Gallery
        self.time_since_update = 0

    def update(self, centroid, label_id, embedding):
        self.current_centroid = centroid
        self.current_label = label_id
        self.features.append(embedding)
        if len(self.features) > 5: self.features.pop(0)
        self.time_since_update = 0

class DeepSORT3D:
    def __init__(self, max_age=3, max_dist=12.0, max_cosine_dist=0.3, lambda_weight=0.6):
        self.tracks = []
        self.next_track_id = 1
        self.max_age, self.max_dist, self.max_cosine_dist, self.lambda_weight = max_age, max_dist, max_cosine_dist, lambda_weight
        self.graph = nx.DiGraph()

    def update(self, frame_T, frame_T1, labels_T, centroids_T1, labels_T1, embeddings_T1, timepoint):
        # 1. Predict motion
        for track in self.tracks:
            track.motion_vector = get_motion_vector(track.current_centroid, track.current_label, labels_T, frame_T, frame_T1)

        unmatched_dets = list(range(len(centroids_T1)))
        matched_pairs = []

        # 2. Matching Cascade (Match newest tracks first)
        for age in range(self.max_age + 1):
            if len(unmatched_dets) == 0: break
            active_tracks = [t for t in self.tracks if t.time_since_update == age]
            if len(active_tracks) == 0: continue

            curr_dets = [centroids_T1[i] for i in unmatched_dets]
            curr_embeds = np.array([embeddings_T1[i] for i in unmatched_dets])

            # Motion Cost
            pred_positions = [t.current_centroid + t.motion_vector for t in active_tracks]
            motion_cost = cdist(np.array(pred_positions), curr_dets, metric='euclidean')

            # Appearance Cost
            app_cost = np.zeros_like(motion_cost)
            for idx, track in enumerate(active_tracks):
                track_feats = np.array(track.features)
                dists = 1.0 - np.dot(curr_embeds, track_feats.T)
                app_cost[idx, :] = np.min(dists, axis=1)

            # Fused Cost & Gates
            fused_cost = self.lambda_weight * (motion_cost / self.max_dist) + (1.0 - self.lambda_weight) * app_cost
            fused_cost[motion_cost > self.max_dist] = 1e5
            fused_cost[app_cost > self.max_cosine_dist] = 1e5

            # Match
            row_ind, col_ind = linear_sum_assignment(fused_cost)
            for r, c in zip(row_ind, col_ind):
                if fused_cost[r, c] < 1e4:
                    matched_pairs.append((active_tracks[r], unmatched_dets[c]))

            # Prune matched from unmatched pool
            matched_det_idx = [det_idx for (_, det_idx) in matched_pairs]
            unmatched_dets = [i for i in unmatched_dets if i not in matched_det_idx]

        # Helper for label extraction
        def get_label(c, labels_img, d_idx):
            cz = int(np.clip(round(c[0]), 0, labels_img.shape[0]-1))
            cy = int(np.clip(round(c[1]), 0, labels_img.shape[1]-1))
            cx = int(np.clip(round(c[2]), 0, labels_img.shape[2]-1))
            lbl = labels_img[cz, cy, cx]
            return lbl if lbl != 0 else (d_idx + 1)

        # 3. Update Tracks
        for track, det_idx in matched_pairs:
            new_c = centroids_T1[det_idx]
            
            # Record edge in graph
            self.graph.add_node(f"T{timepoint}_ID{track.track_id}", t=timepoint, z=new_c[0], y=new_c[1], x=new_c[2])
            self.graph.add_edge(f"T{timepoint-1}_ID{track.track_id}", f"T{timepoint}_ID{track.track_id}")
            
            track.update(new_c, get_label(new_c, labels_T1, det_idx), embeddings_T1[det_idx])

        # 4. Births & Deaths
        for det_idx in unmatched_dets:
            new_c = centroids_T1[det_idx]
            self.graph.add_node(f"T{timepoint}_ID{self.next_track_id}", t=timepoint, z=new_c[0], y=new_c[1], x=new_c[2])
            self.tracks.append(DeepTrack3D(self.next_track_id, new_c, get_label(new_c, labels_T1, det_idx), embeddings_T1[det_idx]))
            self.next_track_id += 1

        matched_tracks = [t for (t, _) in matched_pairs]
        for t in self.tracks:
            if t not in matched_tracks: t.time_since_update += 1
        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]

In [ ]:
# run
# 1. Load Data
zarr_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.zarr"
print(f"Loading data from {zarr_path}...")
z = zarr.open(zarr_path, mode="r")
arr = z['0'] 
MAX_FRAMES = 20

deepsort_tracker = DeepSORT3D(max_age=3, max_dist=12.0, max_cosine_dist=0.3)
print(f"\n--- Starting 3D DeepSORT Tracking ({MAX_FRAMES} frames) ---")

# --- FRAME 0 INIT ---
frame_0_sharp = preprocess_3d_volume(np.asarray(arr[0]))
labels_prev, details_prev = stardist_model.predict_instances(normalize(frame_0_sharp, 1, 99.8, axis=(0,1,2)))
centroids_prev = details_prev['points']
embeddings_prev = extract_cell_embeddings(frame_0_sharp, centroids_prev)

for i, c in enumerate(centroids_prev):
    cz, cy, cx = int(np.clip(round(c[0]), 0, labels_prev.shape[0]-1)), int(np.clip(round(c[1]), 0, labels_prev.shape[1]-1)), int(np.clip(round(c[2]), 0, labels_prev.shape[2]-1))
    lbl = labels_prev[cz, cy, cx] if labels_prev[cz, cy, cx] != 0 else (i + 1)
    
    deepsort_tracker.tracks.append(DeepTrack3D(deepsort_tracker.next_track_id, c, lbl, embeddings_prev[i]))
    deepsort_tracker.graph.add_node(f"T0_ID{deepsort_tracker.next_track_id}", t=0, z=c[0], y=c[1], x=c[2])
    deepsort_tracker.next_track_id += 1
    
print(f"Frame T=0: Spawned {len(centroids_prev)} tracks.")

# --- TIME LOOP ---
frame_prev_sharp = frame_0_sharp

for t in range(1, MAX_FRAMES):
    print(f"Processing Frame T={t}...")
    
    frame_curr_sharp = preprocess_3d_volume(np.asarray(arr[t]))
    labels_curr, details_curr = stardist_model.predict_instances(normalize(frame_curr_sharp, 1, 99.8, axis=(0,1,2)))
    centroids_curr = details_curr['points']
    embeddings_curr = extract_cell_embeddings(frame_curr_sharp, centroids_curr)
    
    deepsort_tracker.update(
        frame_prev_sharp, frame_curr_sharp, labels_prev, 
        centroids_curr, labels_curr, embeddings_curr, t
    )
    
    labels_prev = labels_curr
    frame_prev_sharp = frame_curr_sharp

print("\n==========================================")
print("3D DeepSORT Tracking Complete!")
print(f"Total Detections (Nodes): {deepsort_tracker.graph.number_of_nodes()}")
print(f"Total Connections (Edges): {deepsort_tracker.graph.number_of_edges()}")
print("==========================================")

In [ ]:
# eval not correct, just timepass
from geff import read
from scipy.spatial import KDTree

print("Loading Ground Truth .geff graph...")
geff_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.geff"
gt_graph, metadata = read(geff_path, backend="networkx")

# 1. Extract Ground Truth nodes for t < MAX_FRAMES
gt_nodes_by_time = {t: [] for t in range(MAX_FRAMES)}
for node, data in gt_graph.nodes(data=True):
    t = data.get('t', data.get('time', None))
    if t is not None and t < MAX_FRAMES:
        gt_nodes_by_time[t].append((node, np.array([data['z'], data['y'], data['x']])))

# 2. Extract DeepSORT predicted nodes
pred_nodes_by_time = {t: [] for t in range(MAX_FRAMES)}
for node, data in deepsort_tracker.graph.nodes(data=True):
    t = data.get('t')
    if t is not None and t < MAX_FRAMES and 'z' in data:
        pred_nodes_by_time[t].append((node, np.array([data['z'], data['y'], data['x']])))

# 3. Match GT cells to predictions (16 voxel tolerance for border cells)
gt_to_pred_map = {}
MATCH_TOLERANCE = 16.0 

for t in range(MAX_FRAMES):
    if len(gt_nodes_by_time[t]) == 0 or len(pred_nodes_by_time[t]) == 0:
        continue
    pred_ids, pred_coords = zip(*pred_nodes_by_time[t])
    tree = KDTree(pred_coords)
    
    for gt_id, gt_coord in gt_nodes_by_time[t]:
        dist, idx = tree.query(gt_coord)
        if dist <= MATCH_TOLERANCE:
            gt_to_pred_map[gt_id] = pred_ids[idx]

print(f"Matched {len(gt_to_pred_map)} ground truth cell observations to DeepSORT detections.")

# 4. Check edge agreement (Did DeepSORT draw the same links?)
correct_edges = 0
total_gt_edges = 0

for u, v in gt_graph.edges():
    u_time = gt_graph.nodes[u].get('t', gt_graph.nodes[u].get('time', None))
    v_time = gt_graph.nodes[v].get('t', gt_graph.nodes[v].get('time', None))
    
    if u_time is not None and v_time is not None and max(u_time, v_time) < MAX_FRAMES:
        total_gt_edges += 1
        if u in gt_to_pred_map and v in gt_to_pred_map:
            pred_u = gt_to_pred_map[u]
            pred_v = gt_to_pred_map[v]
            if deepsort_tracker.graph.has_edge(pred_u, pred_v):
                correct_edges += 1

print("\n==========================================")
print("      3D DeepSORT BENCHMARK REPORT        ")
print("==========================================")
print(f"Total Ground Truth Edges (t < {MAX_FRAMES}): {total_gt_edges}")
print(f"Correctly Tracked Edges: {correct_edges}")

if total_gt_edges > 0:
    recall = (correct_edges / total_gt_edges) * 100
    print(f"Tracking Recall: {recall:.2f}%")
print("==========================================")

In [ ]:
# view
import matplotlib.pyplot as plt

# 1. Collect points per frame
gt_coords_by_t = {t: [] for t in range(MAX_FRAMES)}
for node, data in gt_graph.nodes(data=True):
    t = data.get('t', data.get('time', None))
    if t is not None and t < MAX_FRAMES:
        gt_coords_by_t[t].append((data['x'], data['y']))

pred_coords_by_t = {t: [] for t in range(MAX_FRAMES)}
for node, data in deepsort_tracker.graph.nodes(data=True):
    t = data.get('t')
    if t is not None and t < MAX_FRAMES and 'x' in data:
        pred_coords_by_t[t].append((data['x'], data['y']))

# 2. Render 5-Frame Gallery
fig, axes = plt.subplots(1, MAX_FRAMES, figsize=(4 * MAX_FRAMES, 5))
if MAX_FRAMES == 1: axes = [axes]

for t in range(MAX_FRAMES):
    vol = np.asarray(arr[t])
    mip_xy = vol.max(axis=0)
    
    lo, hi = np.percentile(mip_xy, [1, 99.5])
    mip_norm = np.clip((mip_xy - lo) / (hi - lo + 1e-8), 0, 1)
    
    ax = axes[t]
    ax.imshow(mip_norm, cmap='gray')
    ax.set_title(f"Timepoint T = {t}", fontsize=12, fontweight='bold')
    ax.axis('off')
    
    # Plot Detections
    if len(pred_coords_by_t[t]) > 0:
        px, py = zip(*pred_coords_by_t[t])
        ax.scatter(px, py, c='cyan', s=14, alpha=0.7, label='DeepSORT' if t == 0 else "")
        
    # Plot Ground Truth
    if len(gt_coords_by_t[t]) > 0:
        gx, gy = zip(*gt_coords_by_t[t])
        ax.scatter(gx, gy, c='red', s=100, marker='o', edgecolors='white', linewidth=1.5, label='Ground Truth' if t == 0 else "")

axes[0].legend(loc='upper left', framealpha=0.8)
plt.tight_layout()
plt.savefig("/kaggle/working/deepsort_gallery.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# trajectory
# 1. Reconstruct continuous tracks from the graph
track_paths = {}
for node, data in deepsort_tracker.graph.nodes(data=True):
    if '_ID' in node and 't' in data and 'x' in data:
        track_id = node.split('_ID')[1]
        if track_id not in track_paths:
            track_paths[track_id] = []
        track_paths[track_id].append((data['t'], data['z'], data['y'], data['x']))

# Sort points chronologically
for t_id in track_paths:
    track_paths[t_id].sort(key=lambda item: item[0])

# 2. Render Trajectory Plot
fig, ax = plt.subplots(figsize=(10, 10))

# Background: Last frame MIP
vol_last = np.asarray(arr[MAX_FRAMES - 1])
mip_last = vol_last.max(axis=0)
lo, hi = np.percentile(mip_last, [1, 99.5])
ax.imshow(np.clip((mip_last - lo) / (hi - lo + 1e-8), 0, 1), cmap='gray', alpha=0.8)
ax.set_title("Cell Migration Trajectories (DeepSORT 3D)", fontsize=14, fontweight='bold')

colors = plt.cm.rainbow(np.linspace(0, 1, len(track_paths)))
plotted_tracks = 0

for i, (t_id, path) in enumerate(track_paths.items()):
    if len(path) > 1: # Only plot lineages that were successfully tracked
        plotted_tracks += 1
        xs = [p[3] for p in path]
        ys = [p[2] for p in path]
        ax.plot(xs, ys, color=colors[i], linewidth=2.0, alpha=0.8)
        ax.scatter(xs[-1], ys[-1], color=colors[i], s=25, edgecolors='white', linewidth=0.5)

# Overlay Ground Truth Track
gt_track_coords = []
for t in range(MAX_FRAMES):
    if len(gt_coords_by_t[t]) > 0:
        gt_track_coords.append(gt_coords_by_t[t][0])

if len(gt_track_coords) > 1:
    gt_xs, gt_ys = zip(*gt_track_coords)
    ax.plot(gt_xs, gt_ys, color='red', linewidth=3.5, label='Ground Truth Track')
    ax.scatter(gt_xs, gt_ys, color='yellow', s=70, edgecolors='red', linewidth=1.5, zorder=5)

ax.legend(loc='upper left', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig("/kaggle/working/deepsort_trajectories.png", dpi=200, bbox_inches='tight')
plt.show()

print(f"Rendered trajectories for {plotted_tracks} continuous cell tracks!")

In [ ]:
# metrics
track_lengths = [len(path) for path in track_paths.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# 1. Track Length Histogram
ax1.hist(track_lengths, bins=range(1, MAX_FRAMES + 2), align='left', rwidth=0.8, color='teal', edgecolor='black')
ax1.set_title("Track Lifetime Distribution", fontsize=12, fontweight='bold')
ax1.set_xlabel("Track Length (Frames)")
ax1.set_ylabel("Number of Cells")
ax1.grid(axis='y', alpha=0.3)

# 2. Detections per Frame
counts = [len(pred_coords_by_t[t]) for t in range(MAX_FRAMES)]
ax2.plot(range(MAX_FRAMES), counts, marker='o', color='crimson', linewidth=2)
ax2.set_title("Cell Detections per Timepoint", fontsize=12, fontweight='bold')
ax2.set_xlabel("Timepoint T")
ax2.set_ylabel("Active Detections")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# video
import cv2
import numpy as np
import imageio
from IPython.display import Image, display

frames = []

for t in range(MAX_FRAMES):
    # 1. 2D Maximum Intensity Projection
    vol = np.asarray(arr[t])
    mip = vol.max(axis=0)  # Shape (256, 256)
    
    # 2. Normalize to 0-255 uint8
    lo, hi = np.percentile(mip, [1, 99.5])
    mip_norm = np.clip((mip - lo) / (hi - lo + 1e-8), 0, 1)
    mip_uint8 = (mip_norm * 255).astype(np.uint8)
    
    # 3. Convert grayscale to 3-channel RGB image
    img_rgb = cv2.cvtColor(mip_uint8, cv2.COLOR_GRAY2RGB)
    
    # 4. Draw Cyan Dots for DeepSORT Detections (R=0, G=255, B=255)
    if t in pred_coords_by_t:
        for x, y in pred_coords_by_t[t]:
            cv2.circle(img_rgb, (int(round(x)), int(round(y))), radius=2, color=(0, 255, 255), thickness=-1)
            
    # 5. Draw Large Red Ring for Ground Truth (R=255, G=0, B=0)
    if t in gt_coords_by_t:
        for x, y in gt_coords_by_t[t]:
            cv2.circle(img_rgb, (int(round(x)), int(round(y))), radius=6, color=(255, 0, 0), thickness=2)
            
    frames.append(img_rgb)

# Save as a lightweight GIF (~150 KB) and display directly
gif_path = "/kaggle/working/tracker_preview.gif"
imageio.mimsave(gif_path, frames, duration=400, loop=0)

print("Displaying lightweight GIF...")
display(Image(filename=gif_path))

In [ ]:
# 3d
import plotly.graph_objects as go
import numpy as np

# Physical voxel scale from the competition metadata:
# Z resolution is coarser (1.625 µm) compared to X/Y (0.40625 µm)
Z_SCALE = 1.625 / 0.40625  # ~4.0x stretch along Z so cells aren't squashed

fig = go.Figure()

# 1. Reconstruct 3D tracks from deepsort_tracker.graph
track_paths = {}
for node, data in deepsort_tracker.graph.nodes(data=True):
    if '_ID' in node and 't' in data and 'x' in data:
        t_id = node.split('_ID')[1]
        if t_id not in track_paths:
            track_paths[t_id] = []
        track_paths[t_id].append((data['t'], data['z'], data['y'], data['x']))

for t_id in track_paths:
    track_paths[t_id].sort(key=lambda item: item[0])

# 2. Draw 3D Trajectory Lines for DeepSORT predictions
for t_id, path in track_paths.items():
    if len(path) > 1: # Only plot cells tracked for more than 1 frame
        ts = [p[0] for p in path]
        zs = [p[1] * Z_SCALE for p in path]
        ys = [p[2] for p in path]
        xs = [p[3] for p in path]
        
        # Draw the 3D line
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode='lines+markers',
            line=dict(width=4),
            marker=dict(size=3),
            hoverinfo='text',
            text=[f"ID: {t_id} | T={t} | z={z/Z_SCALE:.1f}" for t, z in zip(ts, zs)],
            showlegend=False
        ))

# 3. Draw the Ground Truth 3D Trajectory (Thick Red/Yellow in 3D)
gt_path = []
for t in range(MAX_FRAMES):
    if len(gt_nodes_by_time[t]) > 0:
        # Take the first GT cell
        gt_node, coord = gt_nodes_by_time[t][0]
        gt_path.append((t, coord[0], coord[1], coord[2]))

if len(gt_path) > 1:
    gt_ts = [p[0] for p in gt_path]
    gt_zs = [p[1] * Z_SCALE for p in gt_path]
    gt_ys = [p[2] for p in gt_path]
    gt_xs = [p[3] for p in gt_path]
    
    fig.add_trace(go.Scatter3d(
        x=gt_xs, y=gt_ys, z=gt_zs,
        mode='lines+markers',
        line=dict(color='red', width=10),
        marker=dict(color='yellow', size=8, symbol='diamond'),
        name='Ground Truth 3D Track'
    ))

# 4. Format 3D Scene
fig.update_layout(
    title="<b>Interactive 3D Cell Lineage Tracks (Rotate with your mouse!)</b>",
    scene=dict(
        xaxis_title='X (voxels)',
        yaxis_title='Y (voxels)',
        zaxis_title='Z (scaled µm)',
        aspectmode='data', # Keeps realistic 3D physical proportions
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2) # Default viewing angle
        )
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    height=700
)

fig.show()

In [ ]:
# %% [code]
import glob
import os
import zarr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Z_SCALE = 1.625 / 0.40625  # 4.0x physical Z anisotropy

# 1. Gather all Train and Test .zarr paths
train_paths = sorted(glob.glob("/kaggle/input/competitions/biohub-cell-tracking-during-development/train/*.zarr"))
test_paths = sorted(glob.glob("/kaggle/input/competitions/biohub-cell-tracking-during-development/test/*.zarr"))

# Fallback if path doesn't have 'competitions/'
if len(train_paths) == 0:
    train_paths = sorted(glob.glob("/kaggle/input/biohub-cell-tracking-during-development/train/*.zarr"))
    test_paths = sorted(glob.glob("/kaggle/input/biohub-cell-tracking-during-development/test/*.zarr"))

print(f"Found {len(train_paths)} training samples and {len(test_paths)} test samples.")

results = []

def analyze_sample(path, split_name):
    sample_name = os.path.basename(path).replace(".zarr", "")
    try:
        z_store = zarr.open(path, mode="r")
        vol = np.asarray(z_store['0'][0])  # Inspect Frame 0 (shape: 64, 256, 256)
        
        # Threshold the brightest 1.5% voxels (cell nuclei)
        thresh = np.percentile(vol, 98.5)
        coords = np.argwhere(vol > thresh).astype(np.float32)  # [Z, Y, X]
        
        if len(coords) < 100:
            return None
        
        # Scale Z to true physical proportions
        coords[:, 0] *= Z_SCALE
        
        # 3D PCA: Covariance matrix of cell locations
        coords_centered = coords - coords.mean(axis=0)
        cov = np.cov(coords_centered, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)  # Ascending order
        
        # The smallest eigenvector is the Surface Normal (perpendicular to the sheet)
        normal = eigenvectors[:, 0]  # [n_z, n_y, n_x]
        
        # Tilt angle relative to horizontal XY-plane:
        # 0 deg = flat floor (normal along Z)
        # 90 deg = vertical wall (normal along X or Y)
        # 45 deg = diagonal
        tilt_angle = np.degrees(np.arccos(np.clip(abs(normal[0]), 0, 1)))
        
        # Flatness ratio: thickness / width (smaller = flatter sheet, ~1.0 = solid 3D sphere)
        flatness_ratio = np.sqrt(eigenvalues[0]) / np.sqrt(eigenvalues[1])
        
        is_sheet = "Sheet" if flatness_ratio < 0.45 else "3D Volumetric"
        
        # Determine main slope direction
        abs_ny, abs_nx = abs(normal[1]), abs(normal[2])
        if tilt_angle < 15:
            orientation = "Flat Horizontal"
        elif tilt_angle > 75:
            orientation = "Vertical"
        elif abs_nx > 1.5 * abs_ny:
            orientation = "Diagonal (along X)"
        elif abs_ny > 1.5 * abs_nx:
            orientation = "Diagonal (along Y)"
        else:
            orientation = "Diagonal (slanted X & Y)"

        return {
            "sample": sample_name,
            "split": split_name,
            "tilt_angle_deg": round(tilt_angle, 1),
            "geometry": is_sheet,
            "orientation": orientation,
            "flatness_ratio": round(flatness_ratio, 2),
            "norm_z": round(normal[0], 2),
            "norm_y": round(normal[1], 2),
            "norm_x": round(normal[2], 2)
        }
    except Exception as e:
        print(f"Error on {sample_name}: {e}")
        return None

# Process all Train and Test samples
print("\nAnalyzing training samples...")
for p in train_paths:
    res = analyze_sample(p, "Train")
    if res: results.append(res)

print("Analyzing test samples...")
for p in test_paths:
    res = analyze_sample(p, "Test")
    if res: results.append(res)

df = pd.DataFrame(results)
print(f"\nDone! Successfully analyzed {len(df)} samples.")

In [ ]:
# sample eda
# 1. Show summary table of all samples
pd.set_option('display.max_rows', 100)
display(df[['sample', 'split', 'tilt_angle_deg', 'geometry', 'orientation', 'flatness_ratio']])

# 2. Plot the distribution of Tilt Angles
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of Tilt Angles
ax1.hist(df['tilt_angle_deg'], bins=np.linspace(0, 90, 19), color='royalblue', edgecolor='black', alpha=0.8)
ax1.axvline(df['tilt_angle_deg'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df['tilt_angle_deg'].mean():.1f}°")
ax1.set_title("Distribution of Cell Sheet Tilt Angles across Datasets", fontsize=12, fontweight='bold')
ax1.set_xlabel("Tilt Angle (Degrees from Horizontal)")
ax1.set_ylabel("Number of Samples")
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Geometry breakdown (Sheet vs Volumetric)
geom_counts = df['orientation'].value_counts()
ax2.bar(geom_counts.index, geom_counts.values, color='coral', edgecolor='black', alpha=0.8)
ax2.set_title("Orientations Breakdown", fontsize=12, fontweight='bold')
ax2.set_xlabel("Orientation")
ax2.set_ylabel("Count")
ax2.tick_params(axis='x', rotation=30)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# 3. Key Findings Output
mean_angle = df['tilt_angle_deg'].mean()
std_angle = df['tilt_angle_deg'].std()
print(f"\nAverage Tilt Angle across all samples: {mean_angle:.1f}° ± {std_angle:.1f}°")

In [ ]:
# %% [code]
import numpy as np
import networkx as nx
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

Z_SCALE = 1.625 / 0.40625  # 4.0x physical Z scale

class SlabConstrainedDeepSORT3D:
    def __init__(self, max_age=3, max_dist=14.0, max_cosine_dist=0.35, lambda_weight=0.6, perp_penalty=3.5, max_perp_disp=6.0):
        self.tracks = []
        self.next_track_id = 1
        self.max_age = max_age
        self.max_dist = max_dist
        self.max_cosine_dist = max_cosine_dist
        self.lambda_weight = lambda_weight
        
        # SLAB & PERPENDICULAR CONSTRAINTS
        self.perp_penalty = perp_penalty       # Weight factor penalizing perpendicular movement
        self.max_perp_disp = max_perp_disp     # Hard gate: max allowed movement along normal (µm)
        self.normal = np.array([1.0, 0.0, 0.0]) # Surface normal vector [nz, ny, nx]
        self.d_min = -1e9                      # Slab floor
        self.d_max = 1e9                       # Slab ceiling
        self.graph = nx.DiGraph()

    def fit_slab_geometry(self, centroids):
        """Uses 3D PCA on initial detections to estimate the tissue slab plane and thickness."""
        pts = np.copy(centroids).astype(np.float32)
        pts[:, 0] *= Z_SCALE # Convert to physical µm
        
        pts_centered = pts - pts.mean(axis=0)
        cov = np.cov(pts_centered, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
        
        # Smallest eigenvector is the surface normal
        self.normal = eigenvectors[:, 0]
        
        # Project all points onto the normal axis: d = p · n
        projections = np.dot(pts, self.normal)
        
        # Define slab boundaries (5th to 95th percentile + safety buffer)
        buffer = 15.0 # 15 µm slack
        self.d_min = np.percentile(projections, 2) - buffer
        self.d_max = np.percentile(projections, 98) + buffer
        
        tilt = np.degrees(np.arccos(np.clip(abs(self.normal[0]), 0, 1)))
        print(f"-> Fitted Dynamic Slab: Normal={np.round(self.normal, 2)}, Tilt={tilt:.1f}°, Thickness={self.d_max - self.d_min:.1f} µm")

    def _is_inside_slab(self, coord):
        """Checks if a point physically lies within the tissue thickness."""
        c = np.copy(coord)
        c[0] *= Z_SCALE
        d = np.dot(c, self.normal)
        return self.d_min <= d <= self.d_max

    def _anisotropic_motion_cost(self, pred_pts, det_pts):
        """
        Calculates motion cost penalizing perpendicular movement much more heavily
        than tangential sliding across the sheet.
        """
        N, M = len(pred_pts), len(det_pts)
        cost_matrix = np.zeros((N, M), dtype=np.float32)
        
        for i in range(N):
            # Convert to physical scale
            p_track = np.copy(pred_pts[i])
            p_track[0] *= Z_SCALE
            
            for j in range(M):
                p_det = np.copy(det_pts[j])
                p_det[0] *= Z_SCALE
                
                disp = p_det - p_track
                
                # Decompose into Perpendicular and Tangential
                v_perp = np.dot(disp, self.normal)
                v_tangent = disp - v_perp * self.normal
                
                # Hard gate: cell cannot teleport across the thickness
                if abs(v_perp) > self.max_perp_disp:
                    cost_matrix[i, j] = 1e5
                else:
                    # Anisotropic Distance: Tangential + Weighted Perpendicular
                    cost_matrix[i, j] = np.sqrt(np.sum(v_tangent**2) + self.perp_penalty * (v_perp**2))
                    
        return cost_matrix

    def update(self, frame_T, frame_T1, labels_T, centroids_T1, labels_T1, embeddings_T1, timepoint):
        # 1. Predict locations using motion vectors
        for track in self.tracks:
            track.motion_vector = get_motion_vector(track.current_centroid, track.current_label, labels_T, frame_T, frame_T1)

        # 2. Filter out detections that lie completely outside the biological slab
        valid_det_indices = [i for i, c in enumerate(centroids_T1) if self._is_inside_slab(c)]
        unmatched_dets = valid_det_indices
        matched_pairs = []

        # 3. Matching Cascade (freshest tracks first)
        for age in range(self.max_age + 1):
            if len(unmatched_dets) == 0: break
            active_tracks = [t for t in self.tracks if t.time_since_update == age]
            if len(active_tracks) == 0: continue

            curr_dets = [centroids_T1[i] for i in unmatched_dets]
            curr_embeds = np.array([embeddings_T1[i] for i in unmatched_dets])

            pred_positions = [t.current_centroid + t.motion_vector for t in active_tracks]

            # Motion cost with Bounded Slab Perpendicular Penalty
            motion_cost = self._anisotropic_motion_cost(pred_positions, curr_dets)

            # Appearance Cost (Cosine)
            app_cost = np.zeros_like(motion_cost)
            for idx, track in enumerate(active_tracks):
                track_feats = np.array(track.features)
                dists = 1.0 - np.dot(curr_embeds, track_feats.T)
                app_cost[idx, :] = np.min(dists, axis=1)

            # Fused Cost Matrix
            fused_cost = self.lambda_weight * (motion_cost / self.max_dist) + (1.0 - self.lambda_weight) * app_cost
            fused_cost[motion_cost > self.max_dist] = 1e5
            fused_cost[app_cost > self.max_cosine_dist] = 1e5

            # Hungarian Assignment
            row_ind, col_ind = linear_sum_assignment(fused_cost)
            for r, c in zip(row_ind, col_ind):
                if fused_cost[r, c] < 1e4:
                    matched_pairs.append((active_tracks[r], unmatched_dets[c]))

            matched_det_idx = [det_idx for (_, det_idx) in matched_pairs]
            unmatched_dets = [i for i in unmatched_dets if i not in matched_det_idx]

        def get_label(c, labels_img, d_idx):
            cz = int(np.clip(round(c[0]), 0, labels_img.shape[0]-1))
            cy = int(np.clip(round(c[1]), 0, labels_img.shape[1]-1))
            cx = int(np.clip(round(c[2]), 0, labels_img.shape[2]-1))
            lbl = labels_img[cz, cy, cx]
            return lbl if lbl != 0 else (d_idx + 1)

        # 4. Update Matched Tracks
        for track, det_idx in matched_pairs:
            new_c = centroids_T1[det_idx]
            self.graph.add_node(f"T{timepoint}_ID{track.track_id}", t=timepoint, z=new_c[0], y=new_c[1], x=new_c[2])
            self.graph.add_edge(f"T{timepoint-1}_ID{track.track_id}", f"T{timepoint}_ID{track.track_id}")
            track.update(new_c, get_label(new_c, labels_T1, det_idx), embeddings_T1[det_idx])

        # 5. Births (only for detections inside the slab)
        for det_idx in unmatched_dets:
            new_c = centroids_T1[det_idx]
            self.graph.add_node(f"T{timepoint}_ID{self.next_track_id}", t=timepoint, z=new_c[0], y=new_c[1], x=new_c[2])
            self.tracks.append(DeepTrack3D(self.next_track_id, new_c, get_label(new_c, labels_T1, det_idx), embeddings_T1[det_idx]))
            self.next_track_id += 1

        matched_tracks = [t for (t, _) in matched_pairs]
        for t in self.tracks:
            if t not in matched_tracks: t.time_since_update += 1
        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]

In [ ]:
# %% [code]
# 1. Load Data
zarr_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.zarr"
z = zarr.open(zarr_path, mode="r")
arr = z['0'] 
MAX_FRAMES = 20

slab_tracker = SlabConstrainedDeepSORT3D(max_age=3, max_dist=15.0, perp_penalty=3.0, max_perp_disp=7.0)
print(f"--- Starting Slab-Constrained 3D DeepSORT ({MAX_FRAMES} frames) ---")

# Frame 0 setup
frame_0_sharp = preprocess_3d_volume(np.asarray(arr[0]))
labels_prev, details_prev = stardist_model.predict_instances(normalize(frame_0_sharp, 1, 99.8, axis=(0,1,2)))
centroids_prev = details_prev['points']
embeddings_prev = extract_cell_embeddings(frame_0_sharp, centroids_prev)

# FIT DYNAMIC SLAB
slab_tracker.fit_slab_geometry(centroids_prev)

# Initialize Frame 0 tracks
for i, c in enumerate(centroids_prev):
    if slab_tracker._is_inside_slab(c):
        cz, cy, cx = int(np.clip(round(c[0]), 0, labels_prev.shape[0]-1)), int(np.clip(round(c[1]), 0, labels_prev.shape[1]-1)), int(np.clip(round(c[2]), 0, labels_prev.shape[2]-1))
        lbl = labels_prev[cz, cy, cx] if labels_prev[cz, cy, cx] != 0 else (i + 1)
        
        slab_tracker.tracks.append(DeepTrack3D(slab_tracker.next_track_id, c, lbl, embeddings_prev[i]))
        slab_tracker.graph.add_node(f"T0_ID{slab_tracker.next_track_id}", t=0, z=c[0], y=c[1], x=c[2])
        slab_tracker.next_track_id += 1

print(f"Frame T=0: Spawned {len(slab_tracker.tracks)} valid in-slab tracks.")

# Time Loop
frame_prev_sharp = frame_0_sharp
for t in range(1, MAX_FRAMES):
    print(f"Processing Frame T={t}...")
    frame_curr_sharp = preprocess_3d_volume(np.asarray(arr[t]))
    labels_curr, details_curr = stardist_model.predict_instances(normalize(frame_curr_sharp, 1, 99.8, axis=(0,1,2)))
    centroids_curr = details_curr['points']
    embeddings_curr = extract_cell_embeddings(frame_curr_sharp, centroids_curr)
    
    slab_tracker.update(
        frame_prev_sharp, frame_curr_sharp, labels_prev, 
        centroids_curr, labels_curr, embeddings_curr, t
    )
    
    labels_prev = labels_curr
    frame_prev_sharp = frame_curr_sharp

print("\nTracking Complete!")
print(f"Total Nodes: {slab_tracker.graph.number_of_nodes()} | Total Edges: {slab_tracker.graph.number_of_edges()}")

In [ ]:
# %% [code]
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import imageio
import gc
from IPython.display import Image, display

# 1. Reconstruct track paths from graph
track_paths = {}
for node, data in slab_tracker.graph.nodes(data=True):
    if '_ID' in node and 't' in data and 'x' in data:
        t_id = node.split('_ID')[1]
        if t_id not in track_paths:
            track_paths[t_id] = []
        track_paths[t_id].append((data['t'], data['z'] * Z_SCALE, data['y'], data['x']))

for t_id in track_paths:
    track_paths[t_id].sort(key=lambda item: item[0])

# Collect Ground Truth path
gt_3d_path = []
for t in range(MAX_FRAMES):
    if len(gt_nodes_by_time[t]) > 0:
        _, coord = gt_nodes_by_time[t][0]
        gt_3d_path.append((t, coord[0] * Z_SCALE, coord[1], coord[2]))

# Color palette for tracks
track_ids = list(track_paths.keys())
cmap = plt.cm.turbo(np.linspace(0, 1, len(track_ids)))
id_to_color = {t_id: cmap[i] for i, t_id in enumerate(track_ids)}

print("Rendering 3D Animated Reconstruction (Rotating Camera + Growing Trajectories)...")
movie_frames = []

for t in range(MAX_FRAMES):
    fig = plt.figure(figsize=(9, 9), facecolor='black')
    ax = fig.add_subplot(111, projection='3d', facecolor='black')
    
    # --- A. Volumetric Tissue Context (Downsampled point cloud) ---
    vol = np.asarray(arr[t])[::2, ::4, ::4] # Downsample for speed
    thresh = np.percentile(vol, 96.0)
    tissue_pts = np.argwhere(vol > thresh).astype(np.float32)
    
    # Rescale coords
    tissue_z = tissue_pts[:, 0] * 2 * Z_SCALE
    tissue_y = tissue_pts[:, 1] * 4
    tissue_x = tissue_pts[:, 2] * 4
    
    ax.scatter(tissue_x, tissue_y, tissue_z, c='gray', s=1.0, alpha=0.08, depthshade=True)
    
    # --- B. Draw Growing 3D Trajectory Tails & Active Cells ---
    for t_id, path in track_paths.items():
        # Get history up to current time t
        sub_path = [p for p in path if p[0] <= t]
        
        if len(sub_path) > 0:
            col = id_to_color[t_id]
            xs = [p[3] for p in sub_path]
            ys = [p[2] for p in sub_path]
            zs = [p[1] for p in sub_path]
            
            # Draw tail
            if len(sub_path) > 1:
                ax.plot(xs, ys, zs, color=col, linewidth=2.0, alpha=0.85)
                
            # Draw active cell sphere if present at current frame t
            if sub_path[-1][0] == t:
                ax.scatter(xs[-1], ys[-1], zs[-1], color=col, s=25, edgecolors='white', linewidth=0.5)

    # --- C. Draw Ground Truth Track ---
    gt_sub = [p for p in gt_3d_path if p[0] <= t]
    if len(gt_sub) > 0:
        g_xs = [p[3] for p in gt_sub]
        g_ys = [p[2] for p in gt_sub]
        g_zs = [p[1] for p in gt_sub]
        if len(gt_sub) > 1:
            ax.plot(g_xs, g_ys, g_zs, color='red', linewidth=4.0, zorder=10)
        if gt_sub[-1][0] == t:
            ax.scatter(g_xs[-1], g_ys[-1], g_zs[-1], color='gold', s=100, marker='D', edgecolors='red', linewidth=1.5, zorder=11)

    # --- D. Camera Perspective & Aesthetic Polish ---
    # Slow 3D orbit angle for kinetic depth
    cam_elev = 25
    cam_azim = -60 + (t * 8)  # Orbit by 8 degrees per frame
    ax.view_init(elev=cam_elev, azim=cam_azim)
    
    ax.set_xlim(0, 256)
    ax.set_ylim(0, 256)
    ax.set_zlim(0, 64 * Z_SCALE)
    
    ax.set_title(f"3D Cell Lineage Reconstruction | T = {t}", color='white', fontsize=14, pad=20)
    
    # Hide dark panes for clean sci-fi look
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.grid(False)
    ax.tick_params(colors='gray', labelsize=8)
    
    # Render frame buffer
    fig.canvas.draw()
    frame_rgba = np.asarray(fig.canvas.buffer_rgba())
    movie_frames.append(frame_rgba)
    
    plt.close(fig)
    gc.collect() # Safe memory management
    print(f"Rendered 3D frame {t+1}/{MAX_FRAMES}")

# Save and Display Movie
movie_path = "/kaggle/working/3d_cell_reconstruction.gif"
imageio.mimsave(movie_path, movie_frames, duration=600, loop=0)
print(f"\n3D Movie rendered and saved to {movie_path}!")

display(Image(filename=movie_path))

In [ ]:
# %% [code]
import plotly.graph_objects as go
import numpy as np

Z_SCALE = 1.625 / 0.40625 # ~4.0x physical Z anisotropy

# 1. Reconstruct all track trajectories from slab_tracker.graph
track_paths = {}
for node, data in slab_tracker.graph.nodes(data=True):
    if '_ID' in node and 't' in data and 'x' in data:
        t_id = node.split('_ID')[1]
        if t_id not in track_paths:
            track_paths[t_id] = []
        track_paths[t_id].append((data['t'], data['z'] * Z_SCALE, data['y'], data['x']))

for t_id in track_paths:
    track_paths[t_id].sort(key=lambda item: item[0])

# Collect Ground Truth path
gt_3d_path = []
for t in range(MAX_FRAMES):
    if len(gt_nodes_by_time[t]) > 0:
        _, coord = gt_nodes_by_time[t][0]
        gt_3d_path.append((t, coord[0] * Z_SCALE, coord[1], coord[2]))

# Helper to build traces for any timepoint t
def create_traces_for_timepoint(t):
    lines_x, lines_y, lines_z = [], [], []
    dots_x, dots_y, dots_z = [], [], []
    
    # Pack all trajectories up to time t into a single trace (using None to separate tracks)
    for t_id, path in track_paths.items():
        sub_path = [p for p in path if p[0] <= t]
        if len(sub_path) > 0:
            if len(sub_path) > 1:
                for p in sub_path:
                    lines_x.append(p[3])
                    lines_y.append(p[2])
                    lines_z.append(p[1])
                lines_x.append(None)
                lines_y.append(None)
                lines_z.append(None)
            
            # Active cell at current time t
            if sub_path[-1][0] == t:
                dots_x.append(sub_path[-1][3])
                dots_y.append(sub_path[-1][2])
                dots_z.append(sub_path[-1][1])

    # 1. Trajectory tails
    trace_lines = go.Scatter3d(
        x=lines_x, y=lines_y, z=lines_z,
        mode='lines',
        line=dict(color='cyan', width=3),
        opacity=0.65,
        name='Track Lineages',
        hoverinfo='skip'
    )
    
    # 2. Active cells at time t
    trace_dots = go.Scatter3d(
        x=dots_x, y=dots_y, z=dots_z,
        mode='markers',
        marker=dict(size=4, color='white', symbol='circle', opacity=0.9),
        name='Live Cells',
        hoverinfo='text',
        text=[f"Cell at T={t}" for _ in dots_x]
    )
    
    # 3. Ground Truth track up to t
    gt_sub = [p for p in gt_3d_path if p[0] <= t]
    gt_line_x = [p[3] for p in gt_sub] if len(gt_sub) > 1 else []
    gt_line_y = [p[2] for p in gt_sub] if len(gt_sub) > 1 else []
    gt_line_z = [p[1] for p in gt_sub] if len(gt_sub) > 1 else []
    
    trace_gt_line = go.Scatter3d(
        x=gt_line_x, y=gt_line_y, z=gt_line_z,
        mode='lines',
        line=dict(color='red', width=8),
        name='GT Trajectory'
    )
    
    # 4. Ground Truth cell marker at t
    gt_dot_x = [gt_sub[-1][3]] if len(gt_sub) > 0 and gt_sub[-1][0] == t else []
    gt_dot_y = [gt_sub[-1][2]] if len(gt_sub) > 0 and gt_sub[-1][0] == t else []
    gt_dot_z = [gt_sub[-1][1]] if len(gt_sub) > 0 and gt_sub[-1][0] == t else []
    
    trace_gt_dot = go.Scatter3d(
        x=gt_dot_x, y=gt_dot_y, z=gt_dot_z,
        mode='markers',
        marker=dict(size=9, color='gold', symbol='diamond', line=dict(color='red', width=1.5)),
        name='GT Active Cell'
    )
    
    return [trace_lines, trace_dots, trace_gt_line, trace_gt_dot]

# Build Base Figure (Frame T=0)
initial_traces = create_traces_for_timepoint(0)
fig = go.Figure(data=initial_traces)

# Build Animation Frames (T=0 to T=MAX_FRAMES-1)
frames = []
for t in range(MAX_FRAMES):
    frames.append(go.Frame(
        data=create_traces_for_timepoint(t),
        name=f"T={t}"
    ))
fig.frames = frames

# Setup Interactive Time Slider & Play/Pause Controls
fig.update_layout(
    title="<b>4D Interactive Embryo Viewer (Rotate in 3D + Scrub Time Slider)</b>",
    template="plotly_dark",
    scene=dict(
        xaxis=dict(title='X (voxels)', range=[0, 256], autorange=False),
        yaxis=dict(title='Y (voxels)', range=[0, 256], autorange=False),
        zaxis=dict(title='Z (scaled µm)', range=[0, 64 * Z_SCALE], autorange=False),
        aspectmode='data', # Preserves physical 3D proportions
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.3))
    ),
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "direction": "left",
        "x": 0.1, "y": 0,
        "xanchor": "right", "yanchor": "top",
        "pad": {"r": 10, "t": 50},
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 600, "redraw": True}, "fromcurrent": True, "transition": {"duration": 300}}]
            },
            {
                "label": "❚❚ Pause",
                "method": "animate",
                "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}]
            }
        ]
    }],
    sliders=[{
        "active": 0,
        "x": 0.12, "y": 0,
        "len": 0.85,
        "xanchor": "left", "yanchor": "top",
        "pad": {"b": 10, "t": 50},
        "currentvalue": {
            "font": {"size": 16, "color": "cyan"},
            "prefix": "Timepoint: ",
            "visible": True,
            "xanchor": "right"
        },
        "steps": [
            {
                "args": [[f"T={t}"], {"frame": {"duration": 300, "redraw": True}, "mode": "immediate"}],
                "label": f"T={t}",
                "method": "animate"
            }
            for t in range(MAX_FRAMES)
        ]
    }],
    margin=dict(l=0, r=0, b=0, t=50),
    height=750
)

fig.show()

In [ ]:
# %% [code]
import zarr
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

# 1. Open Zarr store directly (LAZY LOADING - zero RAM footprint!)
zarr_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.zarr"
print(f"Connecting to Zarr store: {zarr_path}")

z_root = zarr.open(zarr_path, mode="r")
z_arr = z_root['0']  # Shape is (T, Z, Y, X), e.g. (100, 64, 256, 256)

T_total, Z_total, Y_total, X_total = z_arr.shape
print(f"Zarr volume dimensions: Time={T_total}, Z={Z_total}, Y={Y_total}, X={X_total}")

Z_SCALE = 1.625 / 0.40625  # ~4.0x physical Z anisotropy

def normalize_slice(img):
    lo, hi = np.percentile(img, [1, 99.8])
    return np.clip((img.astype(np.float32) - lo) / (hi - lo + 1e-8), 0, 1)

# 2. Function called whenever you drag a slider
def inspect_zarr(t=0, z=32, y=128, x=128):
    # FETCH SLICES DIRECTLY FROM DISK CHUNKS
    xy_slice = normalize_slice(z_arr[t, z, :, :])
    xz_slice = normalize_slice(z_arr[t, :, y, :])
    yz_slice = normalize_slice(z_arr[t, :, :, x])
    
    # Value of the exact single voxel under the crosshair
    voxel_val = z_arr[t, z, y, x]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor='#111111')
    
    # 1. XY Plane (Looking from the top)
    axes[0].imshow(xy_slice, cmap='inferno', origin='lower')
    axes[0].axhline(y, color='cyan', linestyle='--', linewidth=1)
    axes[0].axvline(x, color='cyan', linestyle='--', linewidth=1)
    axes[0].set_title(f"XY View (T={t}, Z={z})", color='white', fontsize=13)
    axes[0].set_xlabel("X (voxels)", color='gray')
    axes[0].set_ylabel("Y (voxels)", color='gray')
    
    # 2. XZ Plane (Looking from the side - scaled by Z_SCALE)
    axes[1].imshow(xz_slice, cmap='inferno', origin='lower', aspect=Z_SCALE)
    axes[1].axhline(z, color='cyan', linestyle='--', linewidth=1)
    axes[1].axvline(x, color='cyan', linestyle='--', linewidth=1)
    axes[1].set_title(f"XZ View (Side cut at Y={y})", color='white', fontsize=13)
    axes[1].set_xlabel("X (voxels)", color='gray')
    axes[1].set_ylabel("Z (depth)", color='gray')
    
    # 3. YZ Plane (Looking from the front - scaled by Z_SCALE)
    axes[2].imshow(yz_slice, cmap='inferno', origin='lower', aspect=Z_SCALE)
    axes[2].axhline(z, color='cyan', linestyle='--', linewidth=1)
    axes[2].axvline(y, color='cyan', linestyle='--', linewidth=1)
    axes[2].set_title(f"YZ View (Front cut at X={x})", color='white', fontsize=13)
    axes[2].set_xlabel("Y (voxels)", color='gray')
    axes[2].set_ylabel("Z (depth)", color='gray')
    
    for ax in axes:
        ax.tick_params(colors='gray')
        
    plt.suptitle(f"Voxel Inspector: (T={t}, Z={z}, Y={y}, X={x}) | Raw Intensity: {voxel_val}", color='cyan', fontsize=14)
    plt.tight_layout()
    plt.show()

# 3. Interactive sliders
interact(
    inspect_zarr,
    t=widgets.IntSlider(min=0, max=T_total-1, step=1, value=0, description='Time T:'),
    z=widgets.IntSlider(min=0, max=Z_total-1, step=1, value=Z_total//2, description='Depth Z:'),
    y=widgets.IntSlider(min=0, max=Y_total-1, step=1, value=Y_total//2, description='Height Y:'),
    x=widgets.IntSlider(min=0, max=X_total-1, step=1, value=X_total//2, description='Width X:')
);

In [ ]:
# %% [code]
import scipy.ndimage as ndi
from skimage.feature import peak_local_max
zarr_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.zarr"
z = zarr.open(zarr_path, mode="r")
arr = z['0'] 
def detect_cells_classical_3d(vol_raw, min_distance=(2, 6, 6), threshold_rel=0.18):
    """
    Pure classical 3D cell centroid detector.
    Takes 0.04 seconds, requires NO GPU, NO model weights!
    
    min_distance: (Z, Y, X) minimum spacing in voxels between cell centers.
                  Z is smaller (2) because Z resolution is 4x coarser than Y/X (6).
    """
    vol_float = vol_raw.astype(np.float32)
    
    # 1. 3D Gaussian blur to suppress camera noise (anisotropic sigma)
    vol_smooth = ndi.gaussian_filter(vol_float, sigma=(0.8, 1.8, 1.8))
    
    # 2. Extract true 3D local maxima (the core of each nucleus)
    centroids = peak_local_max(
        vol_smooth,
        min_distance=6,             # Minimum voxel radius
        threshold_rel=threshold_rel, # Reject dark background (<18% max intensity)
        exclude_border=False
    )
    
    return centroids # Returns array of shape (N, 3) -> [Z, Y, X]

# Test on Frame 0
import time
t0 = time.time()
raw_f0 = np.asarray(arr[0])
classical_centroids = detect_cells_classical_3d(raw_f0)
elapsed = time.time() - t0

print(f"Classical 3D Detection finished in: {elapsed*1000:.1f} milliseconds!")
print(f"Total cells found: {len(classical_centroids)}")

In [ ]:
# %% [code]
import plotly.graph_objects as go
import numpy as np

def render_zarr_volume_3d(timepoint=0, downsample=2):
    print(f"Reading Timepoint T={timepoint} directly from Zarr...")
    # Pull ONLY frame T from the Zarr array on disk
    vol_slice = z_arr[timepoint, :, ::downsample, ::downsample].astype(np.float32)
    
    Z_dim, Y_dim, X_dim = vol_slice.shape
    
    # Contrast stretch
    lo, hi = np.percentile(vol_slice, [85, 99.8])
    vol_norm = np.clip((vol_slice - lo) / (hi - lo + 1e-8), 0, 1)
    
    zg, yg, xg = np.mgrid[0:Z_dim, 0:Y_dim, 0:X_dim]
    
    fig = go.Figure(data=go.Volume(
        x=xg.flatten() * downsample,
        y=yg.flatten() * downsample,
        z=(zg * Z_SCALE).flatten(),
        value=vol_norm.flatten(),
        isomin=0.10,
        isomax=0.85,
        opacity=0.12,
        surface_count=20,
        colorscale="Inferno",
        colorbar=dict(title="Raw Intensity", len=0.7)
    ))
    
    fig.update_layout(
        title=f"<b>Raw Zarr 3D Volume | Timepoint T={timepoint}</b>",
        template="plotly_dark",
        scene=dict(
            xaxis_title="X (voxels)",
            yaxis_title="Y (voxels)",
            zaxis_title="Z (scaled µm)",
            aspectmode="data",
            camera=dict(eye=dict(x=1.7, y=1.7, z=1.3))
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        height=700
    )
    # 1. Classical detected coordinates (convert Z to scaled µm)
    cx_z = classical_centroids[:, 0] * Z_SCALE
    cx_y = classical_centroids[:, 1]
    cx_x = classical_centroids[:, 2]
    
    # 2. Add as cyan dots onto your existing 3D figure
    fig.add_trace(go.Scatter3d(
        x=cx_x, y=cx_y, z=cx_z,
        mode='markers',
        marker=dict(size=4, color='cyan', symbol='circle'),
        name='Classical 3D Centroids'
    ))
    fig.show()

# Change timepoint to any frame you want to inspect (e.g., T=0, T=20, T=50)!
render_zarr_volume_3d(timepoint=0)

In [ ]:
# %% [code]
import time
import numpy as np
import zarr
import networkx as nx
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.registration import phase_cross_correlation
import scipy.ndimage as ndi
from skimage.feature import peak_local_max

# 1. Open Zarr volume
zarr_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.zarr"
z_root = zarr.open(zarr_path, mode="r")
arr = z_root['0']
TOTAL_FRAMES = arr.shape[0]  # Full 100 frames!

Z_SCALE = 1.625 / 0.40625  # 4.0x physical Z scale

# ==========================================
# A. Tuned Classical 3D Detector (Catches the faint cells)
# ==========================================
def detect_cells_3d(vol_raw, min_dist=4, thresh_rel=0.14):
    vol_float = vol_raw.astype(np.float32)
    vol_smooth = ndi.gaussian_filter(vol_float, sigma=(0.8, 1.8, 1.8))
    centroids = peak_local_max(
        vol_smooth,
        min_distance=min_dist,
        threshold_rel=thresh_rel,
        exclude_border=False
    )
    return centroids

# ==========================================
# B. Mask-Free 3D Motion Vector Extractor
# ==========================================
# Precompute a 3D Hann bell window to isolate center cell
R = 8
hann_1d = np.hanning(2 * R)
hann_3d = hann_1d[:, None, None] * hann_1d[None, :, None] * hann_1d[None, None, :]

def get_motion_vector_fast(centroid, frame_T, frame_T1, r=8):
    z, y, x = int(centroid[0]), int(centroid[1]), int(centroid[2])
    Z_max, Y_max, X_max = frame_T.shape

    z_min, z_max = max(0, z-r), min(Z_max, z+r)
    y_min, y_max = max(0, y-r), min(Y_max, y+r)
    x_min, x_max = max(0, x-r), min(X_max, x+r)

    crop_T = frame_T[z_min:z_max, y_min:y_max, x_min:x_max].copy()
    crop_T1 = frame_T1[z_min:z_max, y_min:y_max, x_min:x_max]

    if crop_T.shape == (2*r, 2*r, 2*r) and crop_T1.shape == (2*r, 2*r, 2*r):
        # Windowing: Tapers boundary to 0, focusing correlation purely on the center cell
        crop_T_windowed = crop_T * hann_3d
        shift, _, _ = phase_cross_correlation(crop_T_windowed, crop_T1, upsample_factor=5)
        return shift
    else:
        return np.array([0.0, 0.0, 0.0])

# ==========================================
# C. Fast Classical 3D Tracker
# ==========================================
class FastCellTracker3D:
    def __init__(self, max_dist=12.0, max_age=3):
        self.tracks = {} # track_id -> list of (t, z, y, x)
        self.active_tracks = {} # track_id -> current_centroid
        self.track_ages = {}
        self.motion_vectors = {}
        self.next_id = 1
        self.max_dist = max_dist
        self.max_age = max_age
        self.graph = nx.DiGraph()

    def update(self, frame_T, frame_T1, new_centroids, timepoint):
        track_ids = list(self.active_tracks.keys())
        predicted_pos = []

        # 1. Predict
        for tid in track_ids:
            curr_pos = self.active_tracks[tid]
            mv = get_motion_vector_fast(curr_pos, frame_T, frame_T1)
            self.motion_vectors[tid] = mv
            predicted_pos.append(curr_pos + mv)

        matched_pairs = []
        unmatched_dets = list(range(len(new_centroids)))

        # 2. Match
        if len(track_ids) > 0 and len(new_centroids) > 0:
            cost_matrix = cdist(np.array(predicted_pos), new_centroids, metric='euclidean')
            row_ind, col_ind = linear_sum_assignment(cost_matrix)

            for r, c in zip(row_ind, col_ind):
                if cost_matrix[r, c] < self.max_dist:
                    matched_pairs.append((track_ids[r], c))
                    unmatched_dets.remove(c)

        # 3. Update Matched
        matched_ids = []
        for tid, det_idx in matched_pairs:
            new_pos = new_centroids[det_idx]
            self.tracks[tid].append((timepoint, new_pos[0], new_pos[1], new_pos[2]))
            self.active_tracks[tid] = new_pos
            self.track_ages[tid] = 0
            matched_ids.append(tid)

            # Record in graph
            self.graph.add_node(f"T{timepoint}_ID{tid}", t=timepoint, z=new_pos[0], y=new_pos[1], x=new_pos[2])
            self.graph.add_edge(f"T{timepoint-1}_ID{tid}", f"T{timepoint}_ID{tid}")

        # 4. Handle Deaths
        for tid in track_ids:
            if tid not in matched_ids:
                self.track_ages[tid] += 1
                if self.track_ages[tid] > self.max_age:
                    del self.active_tracks[tid]

        # 5. Handle Births
        for det_idx in unmatched_dets:
            new_pos = new_centroids[det_idx]
            tid = self.next_id
            self.tracks[tid] = [(timepoint, new_pos[0], new_pos[1], new_pos[2])]
            self.active_tracks[tid] = new_pos
            self.track_ages[tid] = 0
            self.graph.add_node(f"T{timepoint}_ID{tid}", t=timepoint, z=new_pos[0], y=new_pos[1], x=new_pos[2])
            self.next_id += 1

# ==========================================
# D. Run Across the Full Video
# ==========================================
tracker = FastCellTracker3D(max_dist=12.0)

print(f"Starting Full Video Tracking ({TOTAL_FRAMES} frames)...")
start_time = time.time()

# Frame 0 Init
frame_prev = np.asarray(arr[0]).astype(np.float32)
centroids_0 = detect_cells_3d(frame_prev)

for c in centroids_0:
    tid = tracker.next_id
    tracker.tracks[tid] = [(0, c[0], c[1], c[2])]
    tracker.active_tracks[tid] = c
    tracker.track_ages[tid] = 0
    tracker.graph.add_node(f"T0_ID{tid}", t=0, z=c[0], y=c[1], x=c[2])
    tracker.next_id += 1

print(f"T=0: Initialized with {len(centroids_0)} cells.")

# Tracking Loop for all 100 Frames
for t in range(1, TOTAL_FRAMES):
    frame_curr = np.asarray(arr[t]).astype(np.float32)
    centroids_curr = detect_cells_3d(frame_curr)
    
    tracker.update(frame_prev, frame_curr, centroids_curr, t)
    frame_prev = frame_curr

    if t % 10 == 0 or t == TOTAL_FRAMES - 1:
        fps = t / (time.time() - start_time)
        print(f"Processed Frame {t}/{TOTAL_FRAMES} | Running Speed: {fps:.1f} FPS | Active Tracks: {len(tracker.active_tracks)}")

total_time = time.time() - start_time
print("\n==========================================")
print(f"FINISHED FULL 100-FRAME VIDEO in {total_time:.1f} seconds! ({TOTAL_FRAMES/total_time:.1f} FPS)")
print(f"Total Lineages Discovered: {len(tracker.tracks)}")
print(f"Total Node Detections: {tracker.graph.number_of_nodes()}")
print(f"Total Temporal Edges: {tracker.graph.number_of_edges()}")
print("==========================================")

In [ ]:
# %% [code]
# Detection-only eval: node matching against GT, no tracking involved.
from geff import read
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
import numpy as np

SCALE = np.array([1.625, 0.40625, 0.40625])  # (z, y, x) microns per voxel
MAX_DIST_UM = 7.0
A_PENALTY = 0.1

geff_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.geff"
gt_graph, metadata = read(geff_path, backend="networkx")
T_true = metadata.extra["estimated_number_of_nodes"]

zarr_path = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.zarr"
z_root = zarr.open(zarr_path, mode="r")
arr = z_root['0']
TOTAL_FRAMES = arr.shape[0]

# --- 1. Run detection only, per frame, no linking ---
pred_by_t = {}
T_pred = 0
for t in range(TOTAL_FRAMES):
    vol = np.asarray(arr[t]).astype(np.float32)
    centroids = detect_cells_3d(vol)  # your existing detector
    if len(centroids) == 0:
        continue
    coords_um = centroids * SCALE  # (z, y, x) -> microns
    pred_by_t[t] = list(enumerate(coords_um))  # id = index within frame, unique enough per-t
    T_pred += len(centroids)
    if t % 10 == 0:
        print(f"T={t}: {len(centroids)} detections")

# --- 2. Group GT nodes by timepoint, in physical microns ---
gt_by_t = {}
for node, data in gt_graph.nodes(data=True):
    t = data.get('t', data.get('time'))
    if t is None:
        continue
    coord = np.array([data['z'], data['y'], data['x']]) * SCALE
    gt_by_t.setdefault(t, []).append((node, coord))

gt_t_range = (min(gt_by_t), max(gt_by_t))
print(f"\nGT timepoints:   {gt_t_range[0]}-{gt_t_range[1]} ({len(gt_by_t)} frames)")
print(f"Pred timepoints: 0-{TOTAL_FRAMES-1} ({len(pred_by_t)} frames with detections)")

# --- 3. Per-timepoint bipartite node matching, capped at 7 um ---
matched_gt = 0
total_gt = 0
matched_pred = 0

for t in gt_by_t:
    total_gt += len(gt_by_t[t])
    if t not in pred_by_t:
        continue
    gt_ids, gt_coords = zip(*gt_by_t[t])
    pred_ids, pred_coords = zip(*pred_by_t[t])
    cost = cdist(np.array(gt_coords), np.array(pred_coords))

    row_ind, col_ind = linear_sum_assignment(cost)
    for r, c in zip(row_ind, col_ind):
        if cost[r, c] <= MAX_DIST_UM:
            matched_gt += 1

# --- 4. Node-count penalty (the only place raw over-detection is scored) ---
penalty_term = max(0.0, 1 - A_PENALTY * (T_pred - T_true) / T_true)

print("\n==========================================")
print("         DETECTION-ONLY EVAL                ")
print("==========================================")
print(f"GT nodes matched: {matched_gt} / {total_gt} ({matched_gt/total_gt*100:.1f}%) within {MAX_DIST_UM} um")
print(f"T_pred / T_true:  {T_pred} / {T_true}")
print(f"Node-count penalty multiplier: {penalty_term:.4f}")
print("==========================================")

In [4]:
# %% [code]
import os
import numpy as np
import zarr
import scipy.ndimage as ndi
from skimage.feature import peak_local_max
from geff import read
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

base = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
if not os.path.isdir(base):
    base = "/kaggle/input/biohub-cell-tracking-during-development/train"

MAX_DIST_UM = 7.0
A_PENALTY = 0.1

# --- DoG detector ---
# sigma_small ~ typical nucleus radius / sqrt(2) (matches LoG-equivalent blob response)
# sigma_large ~ 1.6 * sigma_small (standard DoG approximation of LoG)
# Anisotropic: Z sigma scaled down since Z voxels are physically ~4x larger.
CELL_RADIUS_UM = 3.0  # tune this — approx physical nucleus radius

def make_dog_sigmas(cell_radius_um, z_scale_um, xy_scale_um):
    sigma_small_xy = cell_radius_um / (1.414 * xy_scale_um)
    sigma_small_z = cell_radius_um / (1.414 * z_scale_um)
    sigma_small = (sigma_small_z, sigma_small_xy, sigma_small_xy)
    sigma_large = tuple(s * 1.6 for s in sigma_small)
    return sigma_small, sigma_large

def detect_cells_dog(vol_raw, sigma_small, sigma_large, min_distance, threshold_rel):
    vol_f = vol_raw.astype(np.float32)
    small = ndi.gaussian_filter(vol_f, sigma=sigma_small)
    large = ndi.gaussian_filter(vol_f, sigma=sigma_large)
    dog = small - large
    peaks = peak_local_max(dog, min_distance=min_distance,
                            threshold_rel=threshold_rel, exclude_border=False)
    return peaks, dog

# --- Load the problem sample/frame and check the specific missed node first ---
DIAG_SAMPLE = "44b6_d2f34f90"
t_check = 5
gt_node_z, gt_node_y, gt_node_x = 44, 136, 106  # node 229000000003, missed by peak_local_max

z_root = zarr.open(os.path.join(base, DIAG_SAMPLE + ".zarr"), mode="r")
z_arr = z_root['0']
vol = np.asarray(z_arr[t_check]).astype(np.float32)

gt_graph, metadata = read(os.path.join(base, DIAG_SAMPLE + ".geff"), backend="networkx")
scale_map = {ax.name: ax.scale for ax in metadata.axes if ax.type == "space"}
SCALE = np.array([scale_map["z"], scale_map["y"], scale_map["x"]])

sigma_small, sigma_large = make_dog_sigmas(CELL_RADIUS_UM, scale_map["z"], scale_map["y"])
print(f"DoG sigmas: small={sigma_small}, large={sigma_large}")

dog_centroids, dog_response = detect_cells_dog(vol, sigma_small, sigma_large,
                                                 min_distance=3, threshold_rel=0.05)
print(f"DoG found {len(dog_centroids)} centroids at T={t_check}")

# Did DoG catch the specific node peak_local_max missed?
dog_um = dog_centroids * SCALE
gt_target_um = np.array([gt_node_z, gt_node_y, gt_node_x]) * SCALE
dists = np.linalg.norm(dog_um - gt_target_um, axis=1)
nearest = dists.min()
print(f"\nNode 229000000003: nearest DoG detection is {nearest:.2f} um away "
      f"({'MATCHED' if nearest <= MAX_DIST_UM else 'STILL MISSED'})")

# --- Full recall/count check on this sample, same as before ---
def eval_dog_on_sample(name, sigma_small, sigma_large, min_distance=3, threshold_rel=0.05):
    z_root = zarr.open(os.path.join(base, name + ".zarr"), mode="r")
    arr = z_root['0']
    total_frames = arr.shape[0]

    gt_graph, metadata = read(os.path.join(base, name + ".geff"), backend="networkx")
    T_true = metadata.extra["estimated_number_of_nodes"]
    scale_map = {ax.name: ax.scale for ax in metadata.axes if ax.type == "space"}
    SCALE = np.array([scale_map["z"], scale_map["y"], scale_map["x"]])

    pred_by_t = {}
    T_pred = 0
    for t in range(total_frames):
        v = np.asarray(arr[t]).astype(np.float32)
        centroids, _ = detect_cells_dog(v, sigma_small, sigma_large, min_distance, threshold_rel)
        if len(centroids) == 0:
            continue
        pred_by_t[t] = list(enumerate(centroids * SCALE))
        T_pred += len(centroids)

    gt_by_t = {}
    for node, data in gt_graph.nodes(data=True):
        t = data.get('t', data.get('time'))
        if t is None:
            continue
        coord = np.array([data['z'], data['y'], data['x']]) * SCALE
        gt_by_t.setdefault(t, []).append((node, coord))

    matched_gt, total_gt = 0, 0
    for t in gt_by_t:
        total_gt += len(gt_by_t[t])
        if t not in pred_by_t:
            continue
        gt_ids, gt_coords = zip(*gt_by_t[t])
        pred_ids, pred_coords = zip(*pred_by_t[t])
        cost = cdist(np.array(gt_coords), np.array(pred_coords))
        row_ind, col_ind = linear_sum_assignment(cost)
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] <= MAX_DIST_UM:
                matched_gt += 1

    return matched_gt, total_gt, T_pred, T_true

print(f"\nRunning full-sample DoG eval on {DIAG_SAMPLE}...")
matched, total, T_pred, T_true = eval_dog_on_sample(DIAG_SAMPLE, sigma_small, sigma_large)
recall = matched / total if total else 0.0
over = (T_pred - T_true) / T_true if T_true else 0.0
print(f"DoG recall: {matched}/{total} ({recall*100:.1f}%) | T_pred/T_true: {T_pred}/{T_true} ({over*100:+.1f}%)")
print(f"(peak_local_max baseline on this sample was 34.6% recall, {T_pred}/{T_true if T_true else ''})")

DoG sigmas: small=(1.3056250680013057, 5.222500272005223, 5.222500272005223), large=(2.089000108802089, 8.356000435208356, 8.356000435208356)
DoG found 185 centroids at T=5

Node 229000000003: nearest DoG detection is 1.68 um away (MATCHED)

Running full-sample DoG eval on 44b6_d2f34f90...
DoG recall: 240/283 (84.8%) | T_pred/T_true: 21701/24074 (-9.9%)
(peak_local_max baseline on this sample was 34.6% recall, 21701/24074)


In [8]:
# %% [code]
# --- Cell 1: Validate GPU peak-finder against CPU skimage version (must match closely) ---
import cupy as cp
import cupyx.scipy.ndimage as cundi
import numpy as np
import scipy.ndimage as ndi
from skimage.feature import peak_local_max

def dog_sigmas(radius_um, z_scale, xy_scale):
    s_xy = radius_um / (1.414 * xy_scale)
    s_z = radius_um / (1.414 * z_scale)
    small = (s_z, s_xy, s_xy)
    return small, tuple(s * 1.6 for s in small)

def peak_local_max_gpu(vol_gpu, min_distance, threshold_rel):
    size = 2 * min_distance + 1
    local_max = cundi.maximum_filter(vol_gpu, size=size, mode='constant', cval=-cp.inf)
    thresh_val = threshold_rel * vol_gpu.max()
    mask = (vol_gpu == local_max) & (vol_gpu > thresh_val)
    return cp.argwhere(mask)

small, large = dog_sigmas(3.0, scale_map["z"], scale_map["y"])

test_frame = stack[0]
# CPU reference
s_cpu = ndi.gaussian_filter(test_frame, sigma=small)
l_cpu = ndi.gaussian_filter(test_frame, sigma=large)
dog_cpu = s_cpu - l_cpu
peaks_cpu = peak_local_max(dog_cpu, min_distance=3, threshold_rel=0.05, exclude_border=False)

# GPU version
vol_gpu = cp.asarray(test_frame, dtype=cp.float32)
s_gpu = cundi.gaussian_filter(vol_gpu, sigma=small)
l_gpu = cundi.gaussian_filter(vol_gpu, sigma=large)
dog_gpu = s_gpu - l_gpu
peaks_gpu = cp.asnumpy(peak_local_max_gpu(dog_gpu, min_distance=3, threshold_rel=0.05))

print(f"CPU peaks: {len(peaks_cpu)}, GPU peaks: {len(peaks_gpu)}")

# Match each GPU peak to nearest CPU peak (voxel distance) — should be near-zero offset for true matches
from scipy.spatial.distance import cdist
if len(peaks_cpu) > 0 and len(peaks_gpu) > 0:
    dists = cdist(peaks_gpu, peaks_cpu)
    nearest = dists.min(axis=1)
    print(f"GPU peaks with an exact CPU match (dist=0): {(nearest == 0).sum()} / {len(peaks_gpu)}")
    print(f"GPU peaks with CPU match within 2 voxels:    {(nearest <= 2).sum()} / {len(peaks_gpu)}")
    print(f"Max mismatch distance: {nearest.max():.1f} voxels")
else:
    print("One of the peak sets is empty — check thresholds before trusting the sweep.")

CPU peaks: 216, GPU peaks: 216
GPU peaks with an exact CPU match (dist=0): 216 / 216
GPU peaks with CPU match within 2 voxels:    216 / 216
Max mismatch distance: 0.0 voxels


In [ ]:
# %% [code]
# --- Cell 2: GPU-accelerated checkpointed sweep ---
import os
import glob
import time
import random
import json
import numpy as np
import pandas as pd
import zarr
import cupy as cp
import cupyx.scipy.ndimage as cundi
from geff import read
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

TIME_BUDGET_SEC = 5 * 3600
CHECKPOINT_PATH = "/kaggle/working/dog_sweep_checkpoint.csv"
SEED = 42
N_SAMPLES = 300  # GPU is ~30-40x faster — push this way up vs the CPU plan

MAX_DIST_UM = 7.0

RADIUS_GRID = [2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
THRESH_GRID = [0.02, 0.03, 0.05, 0.07, 0.10, 0.15]
MIN_DIST_GRID = [2, 3, 4]

base = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
if not os.path.isdir(base):
    base = "/kaggle/input/biohub-cell-tracking-during-development/train"

random.seed(SEED)
zarr_paths = sorted(glob.glob(os.path.join(base, "*.zarr")))
all_names = [os.path.basename(p).replace(".zarr", "") for p in zarr_paths]
all_names = [n for n in all_names if os.path.exists(os.path.join(base, n + ".geff"))]
chosen = random.sample(all_names, min(N_SAMPLES, len(all_names)))
print(f"Sweeping over {len(chosen)} samples")

if os.path.exists(CHECKPOINT_PATH):
    done_df = pd.read_csv(CHECKPOINT_PATH)
    done_keys = set(zip(done_df.sample_name, done_df.radius_um, done_df.thresh_rel, done_df.min_dist))
    print(f"Resuming — {len(done_df)} combos already completed.")
else:
    done_df = pd.DataFrame()
    done_keys = set()

def dog_sigmas(radius_um, z_scale, xy_scale):
    s_xy = radius_um / (1.414 * xy_scale)
    s_z = radius_um / (1.414 * z_scale)
    small = (s_z, s_xy, s_xy)
    return small, tuple(s * 1.6 for s in small)

def peak_local_max_gpu(vol_gpu, min_distance, threshold_rel):
    size = 2 * min_distance + 1
    local_max = cundi.maximum_filter(vol_gpu, size=size, mode='constant', cval=-cp.inf)
    thresh_val = threshold_rel * vol_gpu.max()
    mask = (vol_gpu == local_max) & (vol_gpu > thresh_val)
    return cp.argwhere(mask)

def load_sample(name):
    z_root = zarr.open(os.path.join(base, name + ".zarr"), mode="r")
    arr = z_root['0']
    stack = np.asarray(arr[:]).astype(np.float32)

    gt_graph, metadata = read(os.path.join(base, name + ".geff"), backend="networkx")
    T_true = metadata.extra["estimated_number_of_nodes"]
    scale_map = {ax.name: ax.scale for ax in metadata.axes if ax.type == "space"}
    SCALE = np.array([scale_map["z"], scale_map["y"], scale_map["x"]])

    gt_by_t = {}
    for node, data in gt_graph.nodes(data=True):
        t = data.get('t', data.get('time'))
        if t is None:
            continue
        coord = np.array([data['z'], data['y'], data['x']]) * SCALE
        gt_by_t.setdefault(t, []).append((node, coord))

    return stack, scale_map, SCALE, gt_by_t, T_true

results_buffer = []
start_time = time.time()
combos_done = 0
combos_skipped = 0
stop_early = False

for name in chosen:
    if stop_early:
        break
    print(f"\n=== Sample: {name} ===")
    stack, scale_map, SCALE, gt_by_t, T_true = load_sample(name)
    n_frames = stack.shape[0]
    stack_gpu = cp.asarray(stack, dtype=cp.float32)  # whole sample resident on GPU

    for radius in RADIUS_GRID:
        if stop_early:
            break
        small, large = dog_sigmas(radius, scale_map["z"], scale_map["y"])

        # DoG computed once per (sample, radius), kept on GPU, reused across thresh/min_dist
        t0 = time.time()
        dog_gpu_stack = cp.empty_like(stack_gpu)
        for t in range(n_frames):
            s = cundi.gaussian_filter(stack_gpu[t], sigma=small)
            l = cundi.gaussian_filter(stack_gpu[t], sigma=large)
            dog_gpu_stack[t] = s - l
        cp.cuda.Stream.null.synchronize()
        filter_elapsed = time.time() - t0

        for min_dist in MIN_DIST_GRID:
            for thresh in THRESH_GRID:
                key = (name, radius, thresh, min_dist)
                if key in done_keys:
                    combos_skipped += 1
                    continue

                if time.time() - start_time > TIME_BUDGET_SEC:
                    print("Time budget reached — stopping and saving checkpoint.")
                    stop_early = True
                    break

                pred_by_t = {}
                T_pred = 0
                for t in range(n_frames):
                    coords_gpu = peak_local_max_gpu(dog_gpu_stack[t], min_dist, thresh)
                    centroids = cp.asnumpy(coords_gpu)
                    if len(centroids) == 0:
                        continue
                    pred_by_t[t] = list(enumerate(centroids * SCALE))
                    T_pred += len(centroids)

                matched_gt, total_gt = 0, 0
                for t in gt_by_t:
                    total_gt += len(gt_by_t[t])
                    if t not in pred_by_t:
                        continue
                    gt_ids, gt_coords = zip(*gt_by_t[t])
                    pred_ids, pred_coords = zip(*pred_by_t[t])
                    cost = cdist(np.array(gt_coords), np.array(pred_coords))
                    row_ind, col_ind = linear_sum_assignment(cost)
                    for r, c in zip(row_ind, col_ind):
                        if cost[r, c] <= MAX_DIST_UM:
                            matched_gt += 1

                recall = matched_gt / total_gt if total_gt else 0.0
                over = (T_pred - T_true) / T_true if T_true else 0.0
                penalty_mult = max(0.0, 1 - 0.1 * over)
                combined_score = recall * penalty_mult

                results_buffer.append({
                    "sample_name": name, "radius_um": radius, "thresh_rel": thresh,
                    "min_dist": min_dist, "matched_gt": matched_gt, "total_gt": total_gt,
                    "T_pred": T_pred, "T_true": T_true, "recall": recall,
                    "over_pct": over * 100, "penalty_mult": penalty_mult,
                    "combined_score": combined_score
                })
                combos_done += 1

                if len(results_buffer) >= 50:
                    pd.concat([done_df, pd.DataFrame(results_buffer)], ignore_index=True) \
                      .to_csv(CHECKPOINT_PATH, index=False)
                    done_df = pd.read_csv(CHECKPOINT_PATH)
                    results_buffer = []

            if stop_early:
                break
        elapsed_h = (time.time() - start_time) / 3600
        print(f"  radius={radius}: filter {filter_elapsed:.1f}s | "
              f"elapsed {elapsed_h:.2f}h/{TIME_BUDGET_SEC/3600:.1f}h | "
              f"combos: {combos_done} done, {combos_skipped} resumed")

    del stack_gpu, dog_gpu_stack
    cp.get_default_memory_pool().free_all_blocks()

if results_buffer:
    pd.concat([done_df, pd.DataFrame(results_buffer)], ignore_index=True) \
      .to_csv(CHECKPOINT_PATH, index=False)

print(f"\nDone. {combos_done} new combos this run. Results at {CHECKPOINT_PATH}")

Sweeping over 199 samples

=== Sample: 6bba_b693381b ===
  radius=2.0: filter 9.4s | elapsed 0.01h/5.0h | combos: 18 done, 0 resumed
  radius=2.5: filter 9.0s | elapsed 0.01h/5.0h | combos: 36 done, 0 resumed
  radius=3.0: filter 9.0s | elapsed 0.02h/5.0h | combos: 54 done, 0 resumed
  radius=3.5: filter 7.8s | elapsed 0.02h/5.0h | combos: 72 done, 0 resumed
  radius=4.0: filter 5.5s | elapsed 0.02h/5.0h | combos: 90 done, 0 resumed
  radius=5.0: filter 9.6s | elapsed 0.03h/5.0h | combos: 108 done, 0 resumed

=== Sample: 44b6_74d0c52e ===
  radius=2.0: filter 1.0s | elapsed 0.03h/5.0h | combos: 126 done, 0 resumed
  radius=2.5: filter 1.0s | elapsed 0.03h/5.0h | combos: 144 done, 0 resumed
  radius=3.0: filter 1.2s | elapsed 0.03h/5.0h | combos: 162 done, 0 resumed
  radius=3.5: filter 1.5s | elapsed 0.04h/5.0h | combos: 180 done, 0 resumed
  radius=4.0: filter 1.7s | elapsed 0.04h/5.0h | combos: 198 done, 0 resumed
  radius=5.0: filter 2.1s | elapsed 0.04h/5.0h | combos: 216 done, 0 r

In [5]:
# %% [code]
# --- Cell 3: Analyze results (run any time, even mid-sweep) ---
df = pd.read_csv(CHECKPOINT_PATH)

agg = df.groupby(["radius_um", "thresh_rel", "min_dist"]).agg(
    total_matched=("matched_gt", "sum"),
    total_gt=("total_gt", "sum"),
    total_pred=("T_pred", "sum"),
    total_true=("T_true", "sum"),
    n_samples=("sample_name", "nunique")
).reset_index()

agg["recall"] = agg.total_matched / agg.total_gt
agg["over_pct"] = (agg.total_pred - agg.total_true) / agg.total_true * 100
agg["penalty_mult"] = (1 - 0.1 * (agg.total_pred - agg.total_true) / agg.total_true).clip(lower=0)
agg["combined_score"] = agg.recall * agg.penalty_mult

top10 = agg.sort_values("combined_score", ascending=False).head(10)
print(f"Configs tested: {len(agg)} | Total combo-runs logged: {len(df)}")
print(top10[["radius_um","thresh_rel","min_dist","recall","over_pct","penalty_mult","combined_score","n_samples"]]
      .to_string(index=False))

Configs tested: 108 | Total combo-runs logged: 21492
 radius_um  thresh_rel  min_dist   recall  over_pct  penalty_mult  combined_score  n_samples
       2.5        0.03         2 0.950097 18.721674      0.981278        0.932309        199
       2.5        0.02         2 0.952790 21.864326      0.978136        0.931957        199
       2.5        0.05         2 0.944666 15.030316      0.984970        0.930468        199
       2.5        0.03         3 0.938635 11.654674      0.988345        0.927696        199
       2.5        0.02         3 0.941126 14.380935      0.985619        0.927591        199
       2.5        0.07         2 0.938118 11.842860      0.988157        0.927008        199
       2.0        0.03         2 0.966831 41.801822      0.958198        0.926416        199
       2.0        0.03         3 0.955527 30.651558      0.969348        0.926239        199
       3.0        0.02         2 0.929124  3.485988      0.996514        0.925885        199
       3.0       